# A08 y A09

## Realizar regresión lineal común y calcular intervalos de confianza

In [3]:
import statsmodels.api as sm
import pandas as pd

df = pd.read_excel('Motor Trend Car Road Tests.xlsx')


X = df[['hp', 'qsec']]
y = df[['mpg']]

X = sm.add_constant(X)

modelo = sm.OLS(y, X).fit()

print("\n--- Resumen Estadístico Completo ---")
print(modelo.summary())

KeyboardInterrupt: 

## Bootsrap

In [ ]:
import numpy as np
from sklearn.utils import resample

# Bootsrap 
n_iterations = 1000  
n_size = len(df)     
stats = []           


for i in range(n_iterations):
    # Crear una muestra con reemplazo
    df_boot = resample(df, n_samples=n_size)
    
    # Definir variables para esta muestra
    X_boot = df_boot[['hp', 'qsec']]
    y_boot = df_boot['mpg']
    X_boot = sm.add_constant(X_boot)
    
    # Ajustar el modelo y guardar coeficientes
    modelo_boot = sm.OLS(y_boot, X_boot).fit()
    stats.append(modelo_boot.params)

# Convertir a DataFrame
bootstrap_df = pd.DataFrame(stats)

# Calcular Intervalos de Confianza Bootstrap
alpha = 0.95
lower_p = ((1.0 - alpha) / 2.0) * 100
upper_p = (alpha + ((1.0 - alpha) / 2.0)) * 100

#Imprimir los resultados de los intervalos de confianza
print("\n--- Intervalos de Confianza Bootstrap (95%) ---")
for column in bootstrap_df.columns:
    lower = np.percentile(bootstrap_df[column], lower_p)
    upper = np.percentile(bootstrap_df[column], upper_p)
    print(f"{column:5}: [{lower:8.4f}, {upper:8.4f}]")


--- Intervalos de Confianza Bootstrap (95%) ---
const: [ 32.1039,  76.6361]
hp   : [ -0.1218,  -0.0563]
qsec : [ -2.2853,  -0.1280]


## Comparación de resultados

In [ ]:
# Obtener coeficientes e intervalos del modelo original (OLS)
ols_params = modelo.params
ols_conf = modelo.conf_int(alpha=0.05)

# Obtener intervalos del Bootstrap
boot_conf_low = bootstrap_df.quantile(0.025)
boot_conf_high = bootstrap_df.quantile(0.975)
boot_means = bootstrap_df.mean()

# Crear DataFrame comparativo
comparacion = pd.DataFrame({
    'Coef_Original': ols_params,
    'OLS_CI_Lower': ols_conf[0],
    'OLS_CI_Upper': ols_conf[1],
    'Coef_Bootstrap_Mean': boot_means,
    'Boot_CI_Lower': boot_conf_low,
    'Boot_CI_Upper': boot_conf_high
})

# Comparación de resultados
print("--- Comparación: OLS Tradicional vs Bootstrap ---")
print(comparacion)


--- Comparación: OLS Tradicional vs Bootstrap ---
       Coef_Original  OLS_CI_Lower  OLS_CI_Upper  Coef_Bootstrap_Mean  \
const      48.323705     25.614894     71.032516            50.255350   
hp         -0.084593     -0.113089     -0.056097            -0.087555   
qsec       -0.886580     -1.979929      0.206770            -0.976225   

       Boot_CI_Lower  Boot_CI_Upper  
const      32.103878      76.636141  
hp         -0.121780      -0.056297  
qsec       -2.285262      -0.128045  


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [ ]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


## Aggregating

In [ ]:
from sklearn.model_selection import train_test_split

# Agarramos todos los datos numéricos del DataFrame
df_numeric = df.select_dtypes(include=[np.number])

X_all = df_numeric.drop(columns=['mpg'])
y_all = df_numeric['mpg']

# Train test split 80 - 20 
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)


df_train = pd.concat([X_train, y_train], axis=1)


In [ ]:
import statsmodels.api as sm

## Bootsraping para generar 1000 modelos
B = 1000 
n_train_samples = len(df_train)
models_list = []

for i in range(B):
    # Crear una muestra bootstrap (con reemplazo)
    df_boot = resample(df_train, n_samples=n_train_samples, replace=True)
    
    # Separar X e y de la muestra bootstrap
    X_boot = df_boot.drop(columns=['mpg'])
    y_boot = df_boot['mpg']
    
    # Añadir la constante para el intercepto y entrenar el modelo
    X_boot = sm.add_constant(X_boot)
    modelo = sm.OLS(y_boot, X_boot).fit()
    
    # Guardamos el modelo en la lista
    models_list.append(modelo)



In [ ]:
# Preparar X_test (básicamente añadimos la constante)
X_test_with_const = sm.add_constant(X_test, has_constant='add')

# Lista para guardar todas las predicciones
all_predictions = []

# Iteraramos sobre cada uno de los 1000 modelos para predecir sobre X_test
for i, modelo in enumerate(models_list):
    preds = modelo.predict(X_test_with_const)
    all_predictions.append(preds)

# Convertimos la lista de predicciones a un DataFrame de pandas para facilitar los cálculos
predictions_df = pd.DataFrame(all_predictions).T

# Calculamos el promedio de las predicciones de los 1000 modelos para cada observación
y_final = predictions_df.mean(axis=1)

In [ ]:
# Comparación de resultados
print("\n Comparación de las predicciónes (y_predicción)\n")
result_display = pd.DataFrame({'Verdadero mpg': y_test.values, 'Predicción mpg': y_final.values}, index=y_test.index)
print(result_display)


 Resultados de las Predicciones Agregadas (y_predicción)

    Verdadero mpg  Predicción mpg
29           19.7       19.058853
15           10.4       11.519852
24           19.2       16.586305
17           32.4       27.153934
8            22.8       31.228598
9            19.2       18.686773
30           15.0       15.226170


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score

# Carga de datos
df = pd.read_excel("Motor Trend Car Road Tests.xlsx")

X = df.select_dtypes(include=[np.number]).drop(columns=['mpg'])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Lugar de busqueda de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'max_leaf_nodes': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# Configuración de K-Fold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Configuración de GridSearchCV
rf_base = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    scoring='r2', 
    cv=kf,    
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Resultados finales
print(grid_search.best_params_)

# Usamos el mejor modelo para la predicción final
best_rf = grid_search.best_estimator_
test_r2 = r2_score(y_test, best_rf.predict(X_test))

print(f"R2 final en el set de prueba: {test_r2:.4f}")

{'max_depth': 3, 'max_leaf_nodes': 5, 'min_samples_split': 5, 'n_estimators': 50}
R2 final en el set de prueba: 0.8086


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Carga de datos
df = pd.read_excel("Motor Trend Car Road Tests.xlsx")

X = df.select_dtypes(include=[np.number]).drop(columns=['mpg'])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Lugar de busqueda de hiperparámetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'max_leaf_nodes': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# Configuración de K-Fold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Configuración de GridSearchCV
rf_base = GradientBoostingRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    scoring='r2', 
    cv=kf,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Resultados finales
print(grid_search.best_params_)

# Usamos el mejor modelo para la predicción final
best_rf = grid_search.best_estimator_
test_r2 = r2_score(y_test, best_rf.predict(X_test))

print(f"R2 final en el set de prueba: {test_r2:.4f}")

{'max_depth': 3, 'max_leaf_nodes': 10, 'min_samples_split': 10, 'n_estimators': 50}
R2 final en el set de prueba: 0.8411
